In [15]:
import ollama
import numpy as np

In [16]:
EMBEDDING_MODEL='hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
LANGUAGE_MODEL='hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF'

In [17]:
VECTOR_DB = []

def add_chunk_to_database(chunk):
    try:
        response = ollama.embed(model=EMBEDDING_MODEL, input=chunk)

        # Handle both dict-like and object-like Ollama responses
        if isinstance(response, dict):
            embedding = response.get("embedding")
            embeddings = response.get("embeddings")
        else:
            embedding = getattr(response, "embedding", None)
            embeddings = getattr(response, "embeddings", None)

        if embedding is not None:
            embedding_list = embedding
        elif embeddings is not None:
            if isinstance(embeddings, list) and embeddings and isinstance(embeddings[0], list):
                embedding_list = embeddings[0]
            else:
                embedding_list = embeddings
        else:
            raise ValueError(f"Could not find embedding in response: {response}")

        embedding_np = np.array(embedding_list, dtype=np.float32)
        VECTOR_DB.append((chunk, embedding_np))
        print(f"Added chunk with embedding of size {embedding_np.shape} to database.")
    except Exception as e:
        print(f"Error adding chunk to database: {e}")

In [18]:
def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1, dtype=np.float32)
    vec2 = np.array(vec2, dtype=np.float32)
    """Compute the cosine similarity between two vectors."""
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    if norm_vec1 == 0 or norm_vec2 == 0:
        return 0.0
    return dot_product / (norm_vec1 * norm_vec2)

In [19]:
def retrieve(query, top_n=3):
    try:
        response = ollama.embed(model=EMBEDDING_MODEL, input=query)

        if isinstance(response, dict):
            embeddings = response.get("embeddings")
            if embeddings is None:
                embeddings = response.get("embedding")
        else:
            embeddings = getattr(response, "embeddings", None)
            if embeddings is None:
                embeddings = getattr(response, "embedding", None)

        if embeddings is None:
            raise ValueError("No embedding found in the response.")

        query_embedding = np.asarray(
            embeddings[0] if isinstance(embeddings[0], (list, np.ndarray)) else embeddings,
            dtype=np.float32
        )

        ranked = sorted(
            (
                (chunk, cosine_similarity(query_embedding, embedding))
                for chunk, embedding in VECTOR_DB
            ),
            key=lambda item: item[1],
            reverse=True
        )

        return ranked[:top_n]

    except Exception as e:
        print(f"Error retrieving chunks: {e}")
        return []

In [20]:
def main():
    print("Loading dataset...")
    dataset = []

    with open("cats.txt", "r", encoding="utf-8") as file:
        dataset = [line.strip() for line in file if line.strip()]

    print(f"Loaded {len(dataset)} entries")

    print("Indexing dataset (this might take a few minutes)...")
    for i, chunk in enumerate(dataset):
        add_chunk_to_database(chunk)
        print(f"Added chunk {i + 1}/{len(dataset)} to the database")

    while True:
        input_query = input('\nAsk me a question about cats (or type "exit" to quit): ')
        if input_query.lower() == "exit":
            break

        print("\nRetrieving knowledge...")
        retrieved_knowledge = retrieve(input_query, top_n=3)

        print("\nRetrieved information:")
        for chunk, similarity in retrieved_knowledge:
            print(f" - (similarity: {similarity:.2f}) {chunk}")

        context_text = "\n".join([f"- {chunk}" for chunk, _ in retrieved_knowledge])
        instruction_prompt = (
            "You are a helpful chatbot that knows about cats. "
            "Use only the following pieces of context to answer the question. "
            "Don't make up any new information:\n"
            f"{context_text}"
        )

        print("\nGenerating response...")
        stream = ollama.chat(
            model=LANGUAGE_MODEL,
            messages=[
                {"role": "system", "content": instruction_prompt},
                {"role": "user", "content": input_query},
            ],
            stream=True,
        )

        print("\nChatbot Response:")
        for chunk in stream:
            content = chunk.get("message", {}).get("content", "")
            print(content, end="", flush=True)
        print()


main()

Loading dataset...
Loaded 10 entries
Indexing dataset (this might take a few minutes)...
Added chunk with embedding of size (768,) to database.
Added chunk 1/10 to the database
Added chunk with embedding of size (768,) to database.
Added chunk 2/10 to the database
Added chunk with embedding of size (768,) to database.
Added chunk 3/10 to the database
Added chunk with embedding of size (768,) to database.
Added chunk 4/10 to the database
Added chunk with embedding of size (768,) to database.
Added chunk 5/10 to the database
Added chunk with embedding of size (768,) to database.
Added chunk 6/10 to the database
Added chunk with embedding of size (768,) to database.
Added chunk 7/10 to the database
Added chunk with embedding of size (768,) to database.
Added chunk 8/10 to the database
Added chunk with embedding of size (768,) to database.
Added chunk 9/10 to the database
Added chunk with embedding of size (768,) to database.
Added chunk 10/10 to the database



Ask me a question about cats (or type "exit" to quit):  how fast can cat run?



Retrieving knowledge...

Retrieved information:
 - (similarity: 0.82) Cats can travel at approximately 31 mph (49 km) over a short distance.
 - (similarity: 0.72) A cat's heart beats at a rate of 110 to 140 beats per minute, nearly twice as fast as a human's.
 - (similarity: 0.70) Cats can jump up to six times their length.

Generating response...

Chatbot Response:
Cats are agile and can travel short distances quickly, but they don't actually "run" in the same way humans do. They can cover short distances of around 7 mph (11 km/h) over a short distance.

According to the data you provided, cats can reach speeds of up to 31 mph (49 km), but it's worth noting that this is not necessarily their average speed while running or navigating obstacles. Instead, they use a combination of speed and agility to quickly change direction and avoid predators.

It's also worth noting that your statement about a cat's heart beating at a rate of 110-140 beats per minute is actually incorrect - cats' he


Ask me a question about cats (or type "exit" to quit):  exit
